# Committor Model: 1D Overdamped Brownian Dynamics

Trains a non-parametric equilibrium committor (`Committor`) on a long overdamped MD trajectory of a particle in 1D. The order parameter λ is the particle position, and the result is compared with the exact committor.

**Data:** `long_MD/<run>/md_path.npy` (50M frames, column 0 = λ). Runs:
- `longmd_flatbr50M` — flat potential (`FlatWall1D`)
- `longmd_2b_br` — two bumps (`CosBumpSeriesWalls`), V = ½(1 − cos(2π(λ + 0.1)/0.2)) on [−0.1, 0.3], i.e. two 1 kT barriers at λ = 0 and 0.2

Both have harmonic walls, V = k(λ − λ_wall)²/2 with k = 100, outside [−0.2, 0.4].

**Boundaries:** `λ <= -0.1` (A) and `λ >= 0.3` (B), the first and last iSTAR interfaces

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cupy as cp

# Plot styling
plt.rc('font', size=10)
plt.rc('axes', titlesize=14)
plt.rc('axes', labelsize=14)
plt.rc('xtick', labelsize=12)
plt.rc('ytick', labelsize=12)
plt.rc('legend', fontsize=12)

# Custom optimal-RC / committor package
import sys
sys.path.insert(0, '/mnt/0bf0c339-34bb-4500-a5fb-f3c2a863de29/DATA/OP_improvement/optimalrcs/optimalrcs-cupy/')
import optimalrcs, metrics, plots

## 2. Data Loading

Loads the first `n_frames` frames of λ (5M frames fit in GPU memory; the full trajectory is 50M).

In [ ]:
SRC = '/mnt/tw06_biommeda_pastime1/11.2024_StapleTIS_Elias/simulations/long_MD'
run = 'longmd_2b_br'               # or 'longmd_flatbr50M'
n_frames = 5_000_000

lam = np.load(f'{SRC}/{run}/md_path.npy', mmap_mode='r')[:n_frames, 0].astype(np.float64)
lambda_a, lambda_b = -0.1, 0.3
print(f'Frames: {len(lam)}')

## 3. Free Energy Profile (FEP)

FEP along λ: flat for `longmd_flatbr50M`, two barriers for `longmd_2b_br`.

In [ ]:
fig, ax = plt.subplots(1, 1)
plots.plot_fep(ax, cp.asarray(lam), xlabel='$\\lambda$')
plt.show()

## 4. Train Committor (Non-Parametric)

A single equilibrium trajectory, so no `i_traj`/`t_traj` and no history features. The CV `y` is λ itself.

**Parameters:**
| Parameter | Value | Description |
|-----------|-------|-------------|
| `ny` | 6 | Polynomial degree in y |
| `max_iter` | 1500 | Maximum iterations |
| `print_step` | 50 | Metrics every 50 iterations |

**Seeds and warm-up** (`seed_runs.run_seeds`, shared with the other notebooks): the fit is repeated for every seed in `SEEDS` (the seed sets which CV and envelope centre each iteration draws). The first `WARMUP_ITER` iterations of every seed can use their own envelope (`WARMUP_ENVELOPE`), gamma (`WARMUP_GAMMA`) and CV (`WARMUP_CV`: `None` random, otherwise the index of the CV to use); then `ENVELOPE` and `GAMMA` take over (`WARMUP_ITER = 0`: no warm-up). After the warm-up and every `CHUNK` iterations the plots in `PLOTS` are shown here and saved to `<OUT>/seed<s>/it<N>_<plot>.png`, with a progress line. The objects of earlier seeds are freed; `q` is the last seed's fit and `results[seed]['r']` every seed's RC.

In [ ]:
import seed_runs

dt = 2e-4                          # MD time step
SEEDS = [0]                        # one CV: the seed only moves the envelope centres
MAX_ITER = 1500
CHUNK = 500                        # iterations between shown and saved plots
ENVELOPE = 0.01                    # envelope_scale (library default)
GAMMA = 0                          # not used by the equilibrium update (NPq)
WARMUP_ITER = 0                    # optional warm-up (0 = off)
WARMUP_ENVELOPE = ENVELOPE
WARMUP_GAMMA = GAMMA
WARMUP_CV = None                   # only one CV here
PLOTS = ('metrics', 'feps', 'obs_pred')
OUT = f'figures/nb_1d_brownian/{run}'

y = cp.asarray(lam)
results, q = seed_runs.run_seeds(
    optimalrcs,
    make_q=lambda: optimalrcs.Committor(boundary0=lam <= lambda_a,    # state A
                                        boundary1=lam >= lambda_b),   # state B
    comp_y=lambda: y,
    fit_kw=dict(ny=6, print_step=50, save_min_delta_zq=False),
    seeds=SEEDS, max_iter=MAX_ITER, chunk=CHUNK, out_dir=OUT, envelope=ENVELOPE, gamma=GAMMA,
    warmup_iter=WARMUP_ITER, warmup_envelope=WARMUP_ENVELOPE, warmup_gamma=WARMUP_GAMMA,
    plots=PLOTS,
    plot_options=dict(metrics=['cross_entropy', 'max_sd_zq', 'delta_r2'], feps_kw=dict(delta_t_sim=dt)),
)

## 5. Comparison with the Exact Committor

For overdamped dynamics q(λ) = ∫ e^{V/kT} from A up to λ, normalized over A to B. Absorption is only checked at saved frames, so the boundaries are shifted outwards by 0.5826 σ, with σ the step size.

In [ ]:
gamma_md, kT = 5.0, 1.0            # friction and temperature of the MD

def potential(x):
    if run == 'longmd_flatbr50M':
        return np.zeros_like(x)
    m = (x >= -0.1) & (x <= 0.3)
    # 2 bumps
    return np.where(m, 0.5 * (1 - np.cos(2 * np.pi * (x + 0.1) / 0.2)), 0)

s = 0.5826 * np.sqrt(2 * kT / gamma_md * dt)
g = np.linspace(lambda_a - s, lambda_b + s, 20001)
w = np.exp(potential(g) / kT)
c = np.concatenate([[0], np.cumsum((w[1:] + w[:-1]) / 2 * np.diff(g))])

# mean NPq per lambda bin
edges = np.linspace(lambda_a, lambda_b, 101)
mid = (edges[1:] + edges[:-1]) / 2
r = cp.asnumpy(q.r_traj)
idx = np.digitize(lam, edges) - 1
inside = (idx >= 0) & (idx < len(mid))
q_npq = np.bincount(idx[inside], weights=r[inside], minlength=len(mid)) / np.bincount(idx[inside], minlength=len(mid))
q_exact = np.interp(mid, g, c / c[-1])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(mid, q_npq, label='NPq')
ax1.plot(mid, q_exact, 'k--', label='exact')
ax1.set(xlabel='$\\lambda$', ylabel='$q$'); ax1.legend(); ax1.grid()
ax2.plot(mid, q_npq - q_exact)
ax2.set(xlabel='$\\lambda$', ylabel='NPq $-$ exact', title=f'RMS {np.sqrt(np.mean((q_npq - q_exact)**2)):.4f}'); ax2.grid()
plt.show()

## 6. Save Results

Uncomment to save the raw committor trajectory and its natural-coordinate version to disk.

In [ ]:
# Save raw committor values
# np.save('q_traj.npy', cp.asnumpy(q.r_traj))

# Save natural committor
# qn_traj = plots.transform_q2qn(q.r_traj, dt_sim=dt)
# np.save('qn_traj.npy', qn_traj)